In [1]:
import requests
import json
import pprint
import pandas as pd
import hmac
import hashlib
import os
import pprint
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 YaBrowser/19.6.1.153 Yowser/2.5 Safari/537.36'}

In [2]:
#Finding out what the total earnings of the top holders of Tyler market is.
#Then dividing it into holders on yes and no

df = pd.read_csv("tyler_holders_filtered.csv")
len(df)

200

In [3]:
# API Configuration
url = "https://data-api.polymarket.com/v1/leaderboard"

In [4]:
top_200_wallets = df["wallet"].tolist()
top_200_wallets[0]

'0xa7a6cd5399040aa661ee595f4421337d80188117'

In [5]:
#Lets try making a call
test_params = {
    "user": top_200_wallets[0]
}
resp = requests.get(url, params=test_params, headers=headers)
test_data = resp.json()
pprint.pp(test_data)

[{'rank': '1552',
  'proxyWallet': '0xa7a6cd5399040aa661ee595f4421337d80188117',
  'userName': 'KoloJones',
  'xUsername': '',
  'verifiedBadge': False,
  'vol': 0,
  'pnl': 126.19088814800125,
  'profileImage': ''}]


In [6]:
all_earnings_list = []


print(f"Starting extraction for {len(top_200_wallets)} wallets...")

for index, wallet in enumerate(top_200_wallets):
        
    if not wallet or wallet == "0x...":
        print("!!NO WALLET FOUND!!")
        continue
    
    print(f"requesting data for {wallet[:10]}")

    params = {
         "category": "OVERALL",
         "timePeriod": "ALL",
         "orderBy": "PNL",
         "user": wallet
     }
    
    try:
        response = requests.get(url, params=params, headers=headers)
        print(response.status_code)

        if response.status_code == 200:
            data = response.json()
    
        if not data:
            print("NOTHING FOUND. RETURN TO BASE")
            continue
    
       
        for entry in data:
            all_earnings_list.append({
                "profit": entry.get("pnl"),
                "user": entry.get("proxyWallet"),
                "username": entry.get("userName")
            })
    
    except Exception as e:
        print(f"Something went wrong: {e}.")
        continue
df_earnings = pd.DataFrame(all_earnings_list)
print(df_earnings.head())

Starting extraction for 200 wallets...
requesting data for 0xa7a6cd53
200
requesting data for 0xe4de861d
200
requesting data for 0xf44dcf50
200
requesting data for 0x00d1bfdb
200
requesting data for 0x96f795d0
200
requesting data for 0x44fccfc4
200
requesting data for 0xb1ca909e
200
requesting data for 0xc8ab97a9
200
requesting data for 0xfc1490c0
200
requesting data for 0x6404bfa3
200
requesting data for 0x2b9dbf4b
200
requesting data for 0xd8163931
200
requesting data for 0x1f9e15f3
200
requesting data for 0x760907c8
200
requesting data for 0x7e31a5e9
200
requesting data for 0xac4a1fab
200
requesting data for 0x0b644e44
200
requesting data for 0x524db836
200
requesting data for 0x7e35a2a8
200
requesting data for 0xc22e6f36
200
requesting data for 0x5fa44ddb
200
requesting data for 0x0f2bcf46
200
requesting data for 0xf17cd97d
200
requesting data for 0xe6fd1806
200
requesting data for 0xbfbb9bdd
200
requesting data for 0xd5b97d08
200
requesting data for 0x32d1623d
200
requesting data 

In [7]:
df_earnings["wallet"] = df_earnings["user"]
df_earnings.drop(columns="user", inplace=True)
df_earnings

,profit,username,wallet
0,5519.377114,KoloJones,0xa7a6cd5399040aa661ee595f4421337d80188117
1,15081.368235,On.The.Spectrum,0xe4de861d59e174d3d153a5968bc122c66f30c949
2,10147.723216,Certify0148,0xf44dcf50490507a44a6244dc07f50686e71060af
3,1327.460662,placerv2,0x00d1bfdb216e363bdb1a04dc02865a4dd99ec8a4
4,20020.766643,ZaGambla,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae
...,...,...,...
195,-32.556677,,0x57b9acdde2dff37c39ad37f24f3293c8d2330464
196,-27.851524,,0xe6000afcee6ea70d272e3ca3517708add214a907
197,-151.778106,,0xe7505c94c7d5980919ad83b8e614142891094850
198,-9.752003,,0xcc1a5d9160199b0862fbe0870d211974a8d87044


In [8]:
df_tyler = pd.read_csv("tyler_holders_filtered.csv")
df_expanded = pd.merge(df_tyler, df_earnings, on="wallet", how="left").drop(columns="Unnamed: 0")

In [9]:
df_expanded.to_csv("tyler_holders_expanded.csv")